# E25 - Large feature bank + importance selection (Option A)

**Goal.** Replicate the *actual* edge of the last-year top solutions: not hand-picking a
few features (which dilute - see ts_all / tf_all), but generating a **large bank** and
letting **importance-based selection** keep only what helps, then validating the selected
subset against the champion with the standard 10-fold paired-t at the **hardened bar
(t >= 2.5)**.

**Bank = champion base15 x 3 scales (45) + boundary-agnostic families we have never
tried**, taken from the secabird / 2nd-place writeups and ported to our real-time
window-vs-history setting:

- **Hurst exponent** (R/S) - long memory / mean reversion
- **AR(4) coefficients** (Yule-Walker) - richer dependence than ac1/ac2
- **Anderson-Darling** 2-sample statistic - tail-sensitive distribution test
- **Levene** (Brown-Forsythe) + **Fligner-Killeen** - robust variance tests
- **Mood's median** test - robust location
- **Sample entropy** - complexity

> Boundary-relative features from those repos (local window around the *known* break,
> distance-to-true-boundary, variance-ratio-dynamics location) do NOT port - we are
> real-time with an unknown boundary - so they are intentionally excluded.

**How to run.** Flip `SMOKE=1` for a ~1 min plumbing check; leave `SMOKE=0` for the full
10-fold run. Selection uses ExtraTrees Gini importance (no LightGBM dependency).


In [ ]:
import os, math
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
from functools import partial
from scipy.special import ndtri
from scipy.stats import rankdata, anderson_ksamp

# Shared harness (same data split / evaluation as every other experiment).
from _overnight import (
    load_pool, select_picks, build_training, evaluate, paired, SMOKE,
)
from _feat_lib import _feat

BASE_KEYS   = ("spec_entropy",)   # base15 builder key
SCALES      = (60, 130, 250)      # E17 champion multiscale
WINDOW      = 250
AR_ORDER    = 4
SAMPEN_CAP  = 128                 # cap sampen length (O(n^2)) for tractability
EPS         = 1e-12
print("SMOKE =", SMOKE)


In [ ]:
def _subwin(w, s):
    n = len(w)
    return w if s >= n else w[-s:]


# --- new, boundary-agnostic feature families ------------------------------
def _hurst_rs(x):
    """Rescaled-range (R/S) Hurst estimate on the raw window."""
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < 16:
        return 0.5
    y = x - x.mean()
    z = np.cumsum(y)
    R = float(z.max() - z.min())
    S = float(x.std())
    if S < EPS or R < EPS:
        return 0.5
    return math.log(R / S) / math.log(n)


def _ar_coefs(x, p=AR_ORDER):
    """Yule-Walker AR(p) coefficients (numpy Toeplitz solve)."""
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean()
    n = len(x)
    if n <= p + 1:
        return [0.0] * p
    r = np.array([float(np.dot(x[:n - k], x[k:])) / n for k in range(p + 1)])
    if r[0] < EPS:
        return [0.0] * p
    # Toeplitz(r[0..p-1])
    R = np.empty((p, p), dtype=np.float64)
    for i in range(p):
        for j in range(p):
            R[i, j] = r[abs(i - j)]
    try:
        phi = np.linalg.solve(R, r[1:p + 1])
    except np.linalg.LinAlgError:
        phi = np.zeros(p)
    return [float(v) for v in phi]


def _anderson(ws, h_sub):
    """Anderson-Darling 2-sample statistic (window vs subsampled history)."""
    try:
        return float(anderson_ksamp([np.asarray(ws), np.asarray(h_sub)]).statistic)
    except Exception:
        return 0.0


def _levene_bf(ws, h_dev):
    """Brown-Forsythe Levene 2-sample statistic (median-centered abs deviations)."""
    za = np.abs(ws - np.median(ws))
    zb = h_dev
    n1, n2 = len(za), len(zb)
    N = n1 + n2
    if n1 < 2 or n2 < 2:
        return 0.0
    m1, m2 = float(za.mean()), float(zb.mean())
    m = (za.sum() + zb.sum()) / N
    num = (N - 2) * (n1 * (m1 - m) ** 2 + n2 * (m2 - m) ** 2)
    den = float(((za - m1) ** 2).sum() + ((zb - m2) ** 2).sum())
    return float(num / den) if den > EPS else 0.0


def _fligner(ws, h_dev):
    """Fligner-Killeen 2-sample statistic (rank-based normal scores of |x-median|)."""
    za = np.abs(ws - np.median(ws))
    zb = h_dev
    n1, n2 = len(za), len(zb)
    N = n1 + n2
    if n1 < 2 or n2 < 2:
        return 0.0
    a = ndtri(0.5 + rankdata(np.concatenate([za, zb])) / (2.0 * (N + 1)))
    abar = float(a.mean())
    v = float(a.var())
    if v <= EPS:
        return 0.0
    a1 = a[:n1]
    return (n1 * (float(a1.mean()) - abar) ** 2 +
            n2 * (float(a[n1:].mean()) - abar) ** 2) / v


def _mood_median(ws, h_sorted):
    """Mood's median test chi-square (equality of medians, window vs history)."""
    med = float(np.median(np.concatenate([ws, h_sorted])))
    n1, n2 = len(ws), len(h_sorted)
    N = n1 + n2
    T = int(np.sum(ws > med) + np.sum(h_sorted > med))
    if T == 0 or T == N:
        return 0.0
    a_above = int(np.sum(ws > med))
    exp_a = n1 * T / N
    var = n1 * n2 * T * (N - T) / (N * N * (N - 1))
    return (a_above - exp_a) ** 2 / var if var > EPS else 0.0


def _sampen(x, m=2):
    """Sample entropy (m=2, r=0.2*std), on the last SAMPEN_CAP points."""
    x = np.asarray(x, dtype=np.float64)
    if len(x) > SAMPEN_CAP:
        x = x[-SAMPEN_CAP:]
    n = len(x)
    if n < m + 2:
        return 0.0
    r = 0.2 * float(x.std())
    if r <= 0:
        return 0.0

    def _count(mm):
        tmpl = np.array([x[i:i + mm] for i in range(n - mm + 1)])
        c = 0
        for i in range(len(tmpl)):
            d = np.max(np.abs(tmpl - tmpl[i]), axis=1)
            c += int(np.sum(d <= r)) - 1
        return c / 2.0

    B = _count(m)
    A = _count(m + 1)
    if B <= 0 or A <= 0:
        return 0.0
    return -math.log(A / B)


In [ ]:
# --- bank assembly: full vector + aligned names ---------------------------
NEW_NAMES = ["hurst"] + [f"ar{k+1}" for k in range(AR_ORDER)] + [
    "ad", "levene", "fligner", "mood", "sampen",
]

BANK_NAMES = []
for s in SCALES:
    BANK_NAMES += [f"base{i:02d}_{s}" for i in range(15)]
    BANK_NAMES += [f"{nm}_{s}" for nm in NEW_NAMES]


def bank_feat(w, ctx):
    """Full bank vector (order matches BANK_NAMES)."""
    h = ctx["sample_sorted"]
    h_dev = np.abs(h - np.median(h))
    h_sub = h if len(h) <= 100 else h[:: max(1, len(h) // 100)][:100]
    out = []
    for s in SCALES:
        ws = _subwin(w, s)
        out.extend(_feat(ws, ctx, keys=BASE_KEYS))          # 15
        out.append(_hurst_rs(ws))
        out.extend(_ar_coefs(ws))                            # AR_ORDER
        out.append(_anderson(ws, h_sub))
        out.append(_levene_bf(ws, h_dev))
        out.append(_fligner(ws, h_dev))
        out.append(_mood_median(ws, h))
        out.append(_sampen(ws))
    return out


IS_BASE = np.array([nm.startswith("base") for nm in BANK_NAMES])
print(f"bank size = {len(BANK_NAMES)}  (base={int(IS_BASE.sum())}, "
      f"new={int((~IS_BASE).sum())})")


## Selection stage

Build the full-bank design matrix on the training picks, fit ExtraTrees, and rank features
by Gini importance. **Keep all base features**, then select the **new** features whose
importance is at least the *weakest kept base feature* (a principled, self-calibrating
threshold - a new feature has to be at least as useful as something the champion already
relies on).

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

X, yi, y_val, train_ids, folds = load_pool()
picks = select_picks(X, train_ids)
print(f"pool: {len(train_ids)} train, {len(folds)} folds x {len(folds[0])}")

Xtr, ytr = build_training(X, yi, picks, bank_feat, WINDOW)
print("design matrix:", Xtr.shape)

et = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42)
et.fit(Xtr, ytr)
imp = et.feature_importances_

base_min = float(imp[IS_BASE].min())
selected_new = [i for i in range(len(BANK_NAMES))
                if (not IS_BASE[i]) and imp[i] >= base_min]
keep_mask = IS_BASE.copy()
for i in selected_new:
    keep_mask[i] = True

order = np.argsort(imp)[::-1]
print("\ntop-20 by importance:")
for i in order[:20]:
    tag = "base" if IS_BASE[i] else "NEW "
    print(f"  {imp[i]:.4f}  {tag}  {BANK_NAMES[i]}")

print(f"\nweakest base importance = {base_min:.4f}")
print(f"selected NEW features ({len(selected_new)}): "
      f"{[BANK_NAMES[i] for i in selected_new]}")
print(f"final feature count = {int(keep_mask.sum())} "
      f"(45 base + {int(keep_mask.sum()) - 45} new)")


## Validation stage

Run the champion baseline and the selected subset through the identical 10-fold pipeline
and apply the hardened bar. Only `t >= 2.5` with `delta > 0` is worth an LB slot.

In [ ]:
def champion_feat(w, ctx):
    out = []
    for s in SCALES:
        out.extend(_feat(_subwin(w, s), ctx, keys=BASE_KEYS))
    return out


def selected_feat(w, ctx):
    v = np.asarray(bank_feat(w, ctx), dtype=np.float64)
    return v[keep_mask].tolist()


base_aucs, nb = evaluate(champion_feat, WINDOW, X, yi, y_val, picks, folds)
sel_aucs,  ns = evaluate(selected_feat, WINDOW, X, yi, y_val, picks, folds)
md_, dsd, t, _ = paired(sel_aucs, base_aucs)

verdict = ("SHIP(hard)" if md_ > 0 and t >= 2.5 else
           "marginal(HOLD)" if md_ > 0 and t >= 2.0 else
           "weak" if md_ > 0 and t >= 1.5 else "noise")

print(f"champion   nf={nb} mean={base_aucs.mean():.4f}")
print(f"selected   nf={ns} mean={sel_aucs.mean():.4f}  "
      f"delta={md_:+.4f}  t={t:+.2f}  -> {verdict}")


## Interpretation

- **`SHIP(hard)` (t >= 2.5):** a real win. Port the selected new feature(s) into the
  champion notebook, verify bit-exact, local-test, then submit.
- **`marginal` / `weak` / `noise`:** consistent with the representation-saturation finding
  (E19/E21/E23). If even a selected bank of genuinely new families can't clear the bar,
  that is strong evidence we have hit the ceiling of the trailing-window-vs-history
  paradigm, and the champion (57.50% / rank 214) should be banked.

*Note on production cost:* `sampen` (O(n^2)) and the scipy-based tests are fine for this
offline CV, but if any wins, re-implement it O(window) / pure-numpy before wiring into the
per-step `infer()` loop, and re-verify bit-exact against this harness.